# Segment Revenue Extraction
Extracts segment revenue from CNINFO annual report PDFs via Gemini.

**Companies**: 士兰微 (分立器件产品) · 华润微 (产品与方案) · 三安光电 (集成电路产品)

**Run order**: Cell 1 → 2 → 3 → 4 (dry-run) → 5 (write)

In [ ]:
# Cell 1 — Install deps
!pip install -q requests google-cloud-storage google-cloud-secret-manager google-genai
import os, subprocess
os.chdir('/content/analog-pd-dashboard')
print('Working dir:', os.getcwd())

In [ ]:
# Cell 2 — GCP auth (needed for GCS upload + Secret Manager)
from google.colab import auth
auth.authenticate_user()
print('GCP authenticated ✓')

In [ ]:
# Cell 3 — Set credentials
from getpass import getpass

# Paste your CNINFO cookie (input box won't show in plaintext)
os.environ['CNINFO_COOKIE'] = getpass('Paste CNINFO_COOKIE: ')

# Fetch Gemini key from Secret Manager
r = subprocess.run(
    ['gcloud', 'secrets', 'versions', 'access', 'latest',
     '--secret=VITE_GEMINI_API_KEY', '--project=st-china-ai-force'],
    capture_output=True, text=True
)
os.environ['GEMINI_API_KEY'] = r.stdout.strip()
print('CNINFO_COOKIE set ✓')
print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# Cell 4 — Dry run (preview only, no write)
!python fetch_segment_rev_pdf.py --companies Silan 'CR Micro' --dry-run

In [ ]:
# Cell 5 — Write to data.json + commit + push
# Only run after verifying Cell 4 output looks correct
!python fetch_segment_rev_pdf.py --companies Silan 'CR Micro'

!git config user.email 'jania.jiang@gmail.com'
!git config user.name 'Jania Jiang'
!git add data.json
!git commit -m 'data: Silan/CR Micro segment revenue from annual report PDFs (分立器件产品/产品与方案)'
!git push origin claude/code-review-UrrfV